# Global goSPL physiography reconstruction

**Author:** Haoyuan Li

This cookbook adapts the global-map workflow used with the goSPL reconstruction from [Salles et al. (2023), *Hundred million years of landscape dynamics from catchment to global scale*](https://www.science.org/doi/10.1126/science.add2541). The original study uses a high-resolution global dataset. To keep this example small and quick to run, the repository instead includes two coarser 4° NetCDF fixtures that follow the original snapshot layout and share exactly the same cell-centered grid.

The derived dataset retains elevation, cumulative erosion/deposition, precipitation, uplift, and basin identifiers. Channelized discharge and sediment-load fields are intentionally omitted because they do not have an unambiguous coarse-grid average.

In [ ]:
%matplotlib inline

from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm

## Load the reconstruction snapshots

The repository root is located from the current working directory, so the cookbook works when launched from the repository root or from its own directory. Each 4° fixture is loaded independently, just like its corresponding full-resolution goSPL snapshot.

The original reconstruction uses a 0.1° global mesh. For this lightweight cookbook, the included snapshots were downsampled to a 4° mesh.

The original reconstruction dataset provides snapshots at 5 Myr intervals, so this example uses the adjacent 5 Ma and 0 Ma snapshots. The workflow is not restricted to those ages: any adjacent pair, such as 100 Ma and 95 Ma, can be used as long as both snapshots share the same latitude and longitude point set. For a research application, obtain the original high-resolution dataset described in the [paper](https://www.science.org/doi/10.1126/science.add2541), then replace the two fixture paths in the cell below with the local paths to the selected files.

In [ ]:
working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "pyproject.toml").is_file()
)
data_directory = (
    repository_root
    / "notebooks/surface_process/data"
)
younger_snapshot = xr.open_dataset(
    data_directory / "physiography-00Ma-4deg.nc", engine="scipy"
).load()
older_snapshot = xr.open_dataset(
    data_directory / "physiography-05Ma-4deg.nc", engine="scipy"
).load()
required_fields = {"elevation", "erodep", "precipitation", "uplift", "basinID"}
assert (
    required_fields
    == set(younger_snapshot.data_vars)
    == set(older_snapshot.data_vars)
)
np.testing.assert_array_equal(younger_snapshot.longitude, older_snapshot.longitude)
np.testing.assert_array_equal(younger_snapshot.latitude, older_snapshot.latitude)
younger_snapshot

In [ ]:
source_crs = ccrs.PlateCarree()
view_crs = ccrs.Mollweide(central_longitude=-20)
longitude, latitude = np.meshgrid(younger_snapshot.longitude, younger_snapshot.latitude)


def finish_global_axis(axis):
    axis.set_global()
    axis.gridlines(color="0.25", linewidth=0.35, alpha=0.35, linestyle="--")
    axis.spines["geo"].set_edgecolor("0.15")
    axis.spines["geo"].set_linewidth(0.8)

## Present-day physiography

Elevation and bathymetry use the area-weighted 0 Ma `elevation` field. The colormap combines a blue scale for negative values, revealing the seafloor and bathymetric structure, with a land-elevation color scheme for positive values over the continents. The zero-elevation contour approximates the reconstructed coastline.

In [ ]:
positions = np.array([9, 27, 46, 65, 84, 103, 122, 141, 150, 157, 165, 174, 183, 197])
positions = (positions - positions[0]) / (positions[-1] - positions[0])
rgb = np.array([
    [35, 27, 73], [45, 51, 125], [30, 77, 158], [38, 102, 174],
    [46, 128, 197], [50, 155, 217], [72, 180, 230], [81, 202, 254],
    [151, 201, 159], [206, 175, 96], [166, 91, 55], [130, 86, 90],
    [120, 108, 109], [251, 251, 251],
]) / 255.0
elevation_cmap = LinearSegmentedColormap.from_list(
    "reconstructed_elevation", list(zip(positions, rgb)), N=256
)

figure_a = plt.figure(figsize=(9, 6), constrained_layout=True)
axis_a = figure_a.add_subplot(1, 1, 1, projection=view_crs)
topography = axis_a.pcolormesh(
    longitude, latitude, younger_snapshot.elevation / 1e3, transform=source_crs,
    cmap=elevation_cmap, norm=Normalize(vmin=-11, vmax=4),
    shading="auto", rasterized=True,
)
axis_a.contour(
    longitude, latitude, younger_snapshot.elevation, levels=[0], colors="#292929",
    linewidths=0.45, transform=source_crs,
)
finish_global_axis(axis_a)
axis_a.set_title("A  Global physiography at 0 Ma", loc="left", weight="bold")
colorbar_a = figure_a.colorbar(
    topography, ax=axis_a, orientation="horizontal", shrink=0.62, pad=0.035
)
colorbar_a.set_label("Elevation (km)")
plt.show()

## Cumulative erosion and deposition

Negative `erodep` values indicate net erosion and positive values indicate net deposition. The colormap allocates more of its color range to values near zero, making smaller-magnitude features easier to distinguish. It approximately reproduces the erosion and deposition color scale used in the original paper.

In [ ]:
erodep_cmap = LinearSegmentedColormap.from_list(
    "erosion_deposition",
    [
        (0.00, "#253494"), (0.40, "#2c7fb8"), (0.45, "#7fcdbb"),
        (0.50, "#f7f7f7"), (0.55, "#fdae61"), (0.60, "#d73027"),
        (1.00, "#7f0000"),
    ],
)
erodep_norm = TwoSlopeNorm(vmin=-5000, vcenter=0, vmax=5000)
figure_b = plt.figure(figsize=(9, 6), constrained_layout=True)
axis_b = figure_b.add_subplot(1, 1, 1, projection=view_crs)
erodep_image = axis_b.pcolormesh(
    longitude, latitude, younger_snapshot.erodep, transform=source_crs,
    cmap=erodep_cmap, norm=erodep_norm, shading="auto", rasterized=True,
)
axis_b.contour(
    longitude, latitude, younger_snapshot.elevation, levels=[0], colors="#303030",
    linewidths=0.45, transform=source_crs,
)
finish_global_axis(axis_b)
axis_b.set_title(
    "B  Cumulative erosion and deposition at 0 Ma", loc="left", weight="bold"
)
colorbar_b = figure_b.colorbar(
    erodep_image, ax=axis_b, orientation="horizontal", shrink=0.62, pad=0.035
)
colorbar_b.set_label("Erosion (-) / deposition (+) (m)")
plt.show()

## Average erosion and sedimentation rate, 5–0 Ma

Because both snapshots use the same point set, their cumulative fields can be differenced directly.

In [ ]:
rate = (younger_snapshot.erodep - older_snapshot.erodep) / 5.0
rate_limit = float(np.nanpercentile(np.abs(rate), 99))
rate_norm = TwoSlopeNorm(vmin=-rate_limit, vcenter=0, vmax=rate_limit)
rate_figure = plt.figure(figsize=(9, 6), constrained_layout=True)
rate_axis = rate_figure.add_subplot(1, 1, 1, projection=view_crs)
rate_image = rate_axis.pcolormesh(
    longitude, latitude, rate, transform=source_crs, cmap=erodep_cmap,
    norm=rate_norm, shading="auto", rasterized=True,
)
rate_axis.contour(
    longitude, latitude, younger_snapshot.elevation, levels=[0], colors="#303030",
    linewidths=0.45, transform=source_crs,
)
finish_global_axis(rate_axis)
rate_axis.set_title(
    "Average erosion and sedimentation rate, 5–0 Ma", loc="left", weight="bold"
)
rate_colorbar = rate_figure.colorbar(
    rate_image, ax=rate_axis, orientation="horizontal", shrink=0.62, pad=0.035
)
rate_colorbar.set_label("Erosion (-) / sedimentation (+) (m/Myr)")
plt.show()